# Stockmog shared-cash PPO portfolio
Upload `ppo-portfolio-top4.tar` and `ppo-portfolio-top4.sha256.json` to `MyDrive/Stockmog`. This notebook trains **one portfolio policy** using one Rs 50,000 account across BSE, M&M, MAXHEALTH and ONGC. No real trades are placed.

Decisions use common 1-minute candles and all nine timeframe inputs and frozen trend predictions. At the next eligible common candle open, sales execute first, then available cash is split equally across selected purchases. Whole shares only; fees and slippage apply. Missing common minutes are skipped. News and research remain bounded side inputs.

News and research coverage is partial, histories are limited to the frozen-predictor era, and the stocks were selected using earlier results. Evaluation is research only, not a prospective profit guarantee. This model cannot resume the independent-stock policies.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
ARCHIVE = Path('/content/drive/MyDrive/Stockmog/ppo-portfolio-top4.tar')
OUTPUT = Path('/content/drive/MyDrive/Stockmog/ppo-shared-portfolio-affordability-v1')
WORK = Path('/content/stockmog-shared-portfolio')
ALLOW_PARTIAL_NEWS = False  # Review coverage in cell 2 before setting True.
ALLOW_PARTIAL_RESEARCH = False  # Review coverage in cell 2 before setting True.
RUN_FULL_TRAINING = False
RUN_FINAL_TEST = False
STEPS = 262_144
NEWS_LOGIT_LIMIT = 0.05
RESEARCH_LOGIT_LIMIT = 0.15


In [ ]:
import hashlib, json, shutil, tarfile
def file_hash(path):
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(8 * 1024**2), b''):
            h.update(chunk)
    return h.hexdigest()
sidecar = json.loads(ARCHIVE.with_suffix('.sha256.json').read_text())
assert ARCHIVE.stat().st_size == sidecar['bytes'], 'Archive size mismatch'
assert file_hash(ARCHIVE) == sidecar['sha256'], 'Archive checksum mismatch'
WORK.mkdir(parents=True, exist_ok=True)
BUNDLE = WORK / 'stockmog-ppo'
marker = WORK / 'archive-sha256.txt'
if (BUNDLE / 'manifest.json').exists():
    if not marker.exists() or marker.read_text().strip() != sidecar['sha256']:
        raise RuntimeError('Archive changed or extraction is unverified. Choose a fresh WORK directory.')
else:
    assert shutil.disk_usage(WORK).free > ARCHIVE.stat().st_size * 1.15, 'Insufficient local disk'
    with tarfile.open(ARCHIVE) as archive:
        archive.extractall(WORK, filter='data')
    marker.write_text(sidecar['sha256'])
manifest = json.loads((BUNDLE / 'manifest.json').read_text())
assert manifest['status'] == 'complete'
print('Source jobs:', len(manifest['jobs']))
print(json.dumps(json.loads((BUNDLE / 'news-coverage.json').read_text()), indent=2))
print(json.dumps(json.loads((BUNDLE / 'research-coverage.json').read_text()), indent=2))
print('\n'.join(manifest['notes']))


In [ ]:
import os, subprocess, sys
os.environ.setdefault('OMP_NUM_THREADS', '2')
SOURCE = BUNDLE / 'source'
for name, expected in manifest['files'].items():
    if name.startswith('source/'):
        assert file_hash(BUNDLE / name) == expected, f'Source checksum mismatch: {name}'
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(SOURCE / 'requirements-rl.txt')])
sys.path.insert(0, str(SOURCE))
import torch
from data_pipeline.rl_colab_training import PPOConfig
from data_pipeline.rl_portfolio_training import train_portfolio
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
print('Torch:', torch.__version__)


In [ ]:
import numpy as np
from data_pipeline.rl_portfolio_training import SharedPortfolioEnv

# Apply only to the existing archive version; do not overwrite its checked files.
old_source = '380532d39a170fb351a41affb4c2d06499ca70ec0ac22bb71f1baa3c8299cfa4'
if file_hash(BUNDLE / 'source/data_pipeline/rl_portfolio_training.py') != old_source:
    raise RuntimeError('Unexpected portfolio source version; do not apply this patch.')

def affordable_action_masks(self):
    buys = self.actions == 1
    sells = self.actions == 2
    valid_sell = (self.actions != 2) | (self.shares > 0)
    prices = self.closes[self.index]
    proceeds = (sells * self.shares * prices).sum(axis=1) * (1 - self.config.slippage) * (1 - self.config.fee)
    budget = (self.cash + proceeds) / np.maximum(buys.sum(axis=1), 1)
    cost = prices * (1 + self.config.slippage) * (1 + self.config.fee)
    valid_buy = ~buys | ((self.shares == 0) & (budget[:, None] >= cost))
    return np.all(valid_buy & valid_sell, axis=1)

original_step = getattr(SharedPortfolioEnv.step, '_original_portfolio_step', SharedPortfolioEnv.step)
def step_with_rejection_penalty(self, action):
    if not self.action_space.contains(int(action)):
        return original_step(self, action)
    was_valid = bool(self.action_masks()[int(action)])
    previous_rejections = self.rejections
    observation, reward, terminated, truncated, info = original_step(self, action)
    if was_valid and self.rejections > previous_rejections:
        reward -= 0.001
    return observation, reward, terminated, truncated, info
step_with_rejection_penalty._original_portfolio_step = original_step
SharedPortfolioEnv.action_masks = affordable_action_masks
SharedPortfolioEnv.step = step_with_rejection_penalty
print('Affordability fix applied. Use a fresh OUTPUT directory for retraining.')


## Benchmark
Select your GPU and run setup. After reviewing coverage, set both partial-coverage flags to True in cell 1 and rerun it. The benchmark prepares all four stocks and nine timeframes, then trains and validates a temporary portfolio policy; it may take substantially longer than the independent-policy pilot. No final-test results are used.

In [ ]:
from tempfile import TemporaryDirectory
from time import perf_counter
from math import ceil, isfinite
DEVICE = 'cuda'  # Use 'cpu' for a CPU comparison.
CU_PER_HOUR = None  # Enter the runtime's displayed CU/hour.
BENCHMARK_STEPS = 8192
if DEVICE == 'cuda' and not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU found. Select a GPU or set DEVICE to cpu.')
assert manifest['news_complete'] or ALLOW_PARTIAL_NEWS, 'Review news coverage, then allow partial news.'
assert manifest.get('research_complete') or ALLOW_PARTIAL_RESEARCH, 'Review research coverage, then allow partial research.'
if CU_PER_HOUR is not None and (not isfinite(CU_PER_HOUR) or CU_PER_HOUR <= 0):
    raise ValueError('CU_PER_HOUR must be positive, or None.')
pilot_config = PPOConfig(steps=BENCHMARK_STEPS, device=DEVICE, news_logit_limit=NEWS_LOGIT_LIMIT,
                         research_logit_limit=RESEARCH_LOGIT_LIMIT)
with TemporaryDirectory(prefix='portfolio-benchmark-', dir=WORK) as scratch:
    started = perf_counter()
    pilot = train_portfolio(BUNDLE, Path(scratch), pilot_config,
                            allow_partial_news=ALLOW_PARTIAL_NEWS,
                            allow_partial_research=ALLOW_PARTIAL_RESEARCH)
    seconds = perf_counter() - started
speed = pilot['timesteps'] / seconds
rollout_size = pilot_config.rollout * pilot_config.envs
full_steps = ceil(STEPS / rollout_size) * rollout_size
hours = full_steps / speed / 3600
print(f"Portfolio pilot: {pilot['timesteps']:,} steps in {seconds:.1f}s ({speed:.1f} steps/sec)")
print(f'Projected {full_steps:,} training steps: {hours:.1f} active hours')
print('Includes pilot preparation and validation; projection may overstate fixed costs and excludes Drive I/O and interruptions.')
if CU_PER_HOUR is not None:
    print(f'Projected {hours * CU_PER_HOUR:.1f} CU before overhead')


In [ ]:
config = PPOConfig(steps=STEPS, device=DEVICE, news_logit_limit=NEWS_LOGIT_LIMIT,
                   research_logit_limit=RESEARCH_LOGIT_LIMIT)
if RUN_FULL_TRAINING:
    training = train_portfolio(BUNDLE, OUTPUT, config, allow_partial_news=ALLOW_PARTIAL_NEWS,
                               allow_partial_research=ALLOW_PARTIAL_RESEARCH)
    print('Training:', training['status'])
    print('Validation:', training['ppo'])
else:
    print('Full training disabled. Review benchmark, coverage and budget first.')
print('Rerun with unchanged archive/settings to resume verified checkpoints.')


In [ ]:
if RUN_FINAL_TEST:
    final = train_portfolio(BUNDLE, OUTPUT, config, allow_partial_news=ALLOW_PARTIAL_NEWS,
                            allow_partial_research=ALLOW_PARTIAL_RESEARCH, evaluate_test=True)
    import pandas as pd
    results = pd.DataFrame([{'strategy': name, **final[name]} for name in ('ppo', 'buy_hold', 'cash')])
    results.to_csv(OUTPUT / 'test-results.csv', index=False)
    display(results)
else:
    print('Final test disabled. Enable only after fixing the training design.')
